Import libraries and functions

In [1]:
import time
from poisson_bcann_vi import *
from codebase_0 import CANN, MLP, cann_training_loop, mlp_train_loop, DeviceLoader
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from poisson_horseshoe import HorseshoePoissonBCANN

seed = 42
set_seed(seed)
pyro.clear_param_store()

/workspace/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data and perform train/validation/test split

In [ ]:
device = torch.device("cuda")
torch.set_default_device("cpu")

dataset = pd.read_csv("freMTPL2freq_transformed.csv", index_col= 0) 
y = torch.tensor(dataset["ClaimNb"].to_numpy(), dtype=torch.float32)
exposure = torch.tensor(dataset["Exposure"].to_numpy(), dtype=torch.float32)
X = torch.tensor(dataset.drop(columns=["IDpol", "Exposure","ClaimNb"]).to_numpy(), dtype=torch.float32)   # 48 columns

# Learning-test set split
X_learn, X_test, y_learn, y_test, exp_learn, exp_test = train_test_split(
    X, y, exposure, test_size= 0.1, random_state= seed
)

# Training-CV set split
X_train, X_val, y_train, y_val, exp_train, exp_val = train_test_split(
    X_learn, y_learn, exp_learn, test_size= 0.1, random_state= seed
)

learn_scaler = MinMaxScaler().fit(X_learn)
X_learn = learn_scaler.transform(X_learn)
X_test = learn_scaler.transform(X_test)

train_scaler = MinMaxScaler().fit(X_train)
X_train = train_scaler.transform(X_train)
X_val = train_scaler.transform(X_val)

def T(a):
    return torch.as_tensor(a, dtype=torch.float32, device=device)

X_train, X_val, X_learn, X_test = map(T, (X_train, X_val, X_learn, X_test))
y_train, y_val, y_learn, y_test = map(T, (y_train, y_val, y_learn, y_test))
exp_train, exp_val, exp_learn, exp_test = map(T, (exp_train, exp_val, exp_learn, exp_test))

torch.set_default_device(device)

train_loader = DeviceLoader(
        X_train, y_train, torch.log(exp_train),
        batch_size = 10000,
        shuffle= True
)
val_loader = DeviceLoader(
        X_val, y_val, torch.log(exp_val),
        batch_size= 10000
)
learn_loader = DeviceLoader(
        X_learn, y_learn, torch.log(exp_learn),
        batch_size= 10000
)
test_loader = DeviceLoader(
        X_test, y_test, torch.log(exp_test),
        batch_size= 10000
)

Start a HorseshoePoissonBCANN with its variation posterior object

In [11]:
pyro.clear_param_store()

network_architecture = [20, 15, 10]

horseshoe_poisson_cann = HorseshoePoissonBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh},
)

glm_mle = poisson_fit_glm_mle(X_train, y_train, log_exposure= torch.log(exp_train), random_state=seed)
horseshoe_guide = build_guide_horseshoe_bcann(horseshoe_poisson_cann, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)

Searching for best epoch

In [12]:
horseshoe_bcann_history_full = poisson_bcann_svi_training_loop(
        horseshoe_poisson_cann, horseshoe_guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=10, patience=5, random_state=seed, device = device)


Epoch 1 | -ELBO/n: 0.2242 | train deviance: 0.3150 | val deviance: 0.3201
Epoch 50 | -ELBO/n: 0.2098 | train deviance: 0.3085 | val deviance: 0.3140
Epoch 100 | -ELBO/n: 0.2090 | train deviance: 0.3078 | val deviance: 0.3134
Epoch 150 | -ELBO/n: 0.2081 | train deviance: 0.3076 | val deviance: 0.3129
Epoch 200 | -ELBO/n: 0.2077 | train deviance: 0.3074 | val deviance: 0.3129
Epoch 250 | -ELBO/n: 0.2076 | train deviance: 0.3075 | val deviance: 0.3130
Epoch 300 | -ELBO/n: 0.2074 | train deviance: 0.3072 | val deviance: 0.3130


In [15]:
horseshoe_bcann_history_full.get("best_epoch")

280

Retrain with best epoch

In [3]:
pyro.clear_param_store()

network_architecture = [20, 15, 10]

horseshoe_poisson_cann = HorseshoePoissonBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh},
)

glm_mle = poisson_fit_glm_mle(X_train, y_train, log_exposure= torch.log(exp_train), random_state=seed)
horseshoe_guide = build_guide_horseshoe_bcann(horseshoe_poisson_cann, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)
time_0 = time.time()
# set val_every to 280 to make sure no validation loop is performed -> faster
poisson_horseshoe = poisson_bcann_svi_training_loop(
        horseshoe_poisson_cann, horseshoe_guide, train_loader, val_loader,
        n_epochs=280, lr=5e-3, val_every=280, patience=6, random_state=seed, device = device)
time_1 = time.time()
print(time_1 - time_0)

Epoch 1 | -ELBO/n: 0.2242 | train deviance: 0.3150 | val deviance: 0.3201
1262.8959712982178


In [8]:
print(poisson_horseshoe["best_val_deviance"])
print(poisson_horseshoe["train_deviance"][-1])

0.3127731382846832
0.3073206841945648


Train on learn set to get performance on test set

In [13]:
pyro.clear_param_store()

network_architecture = [20, 15, 10]

best_horseshoe_poisson_cann = HorseshoePoissonBCANN(
    in_features= 48,
    neural_net_params={"hidden_activation_nodes": network_architecture,
                        "hidden_activations": nn.Tanh},
)

glm_mle = poisson_fit_glm_mle(X_train, y_train, log_exposure= torch.log(exp_train), random_state=seed)
horseshoe_guide = build_guide_horseshoe_bcann(best_horseshoe_poisson_cann, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)
time_0 = time.time()
# set val_every to 280 to make sure no validation loop is performed -> faster
poisson_horseshoe_best_history = poisson_bcann_svi_training_loop(
        best_horseshoe_poisson_cann, horseshoe_guide, learn_loader, val_loader,
        n_epochs=280, lr=5e-3, val_every=280, patience=6, random_state=seed, device = device)
time_1 = time.time()
print(time_1 - time_0)

Epoch 1 | -ELBO/n: 0.2231 | train deviance: 0.3152 | val deviance: 0.3198
1344.2717401981354


In [ ]:
# write to disk for later uses
torch.save({
    "recipe": {
        "cls": "HorseshoePoissonBCANN",
        "kwargs": dict(in_features= 48,
                        neural_net_params={"hidden_activation_nodes": network_architecture,
                                "hidden_activations": "Tanh"}),
    },
    "guide_builder": "build_guide_horseshoe_bcann",
    "guide_kwargs": dict(rank=10),                     # + tau_init for the horseshoe builders
    "guide_state": {k: v.detach().cpu() for k, v in horseshoe_guide.state_dict().items()},
}, "runs/poisson_horseshoe_bcann_vi.pt")

In [15]:
# Poisson BCANN with horseshoe prior deviance on test set
evaluate_bcann(best_horseshoe_poisson_cann, horseshoe_guide, test_loader, 200, device = device)

0.30298203229904175

# Full epochs run

In [7]:
horseshoe_bcann_history_full = poisson_bcann_svi_training_loop(
        horseshoe_poisson_cann, horseshoe_guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=10, patience=301, random_state=seed, device = device)


Epoch 1 | -ELBO/n: 0.2242 | train deviance: 0.3150 | val deviance: 0.3201
Epoch 50 | -ELBO/n: 0.2098 | train deviance: 0.3085 | val deviance: 0.3140
Epoch 100 | -ELBO/n: 0.2090 | train deviance: 0.3078 | val deviance: 0.3134
Epoch 150 | -ELBO/n: 0.2081 | train deviance: 0.3076 | val deviance: 0.3129
Epoch 200 | -ELBO/n: 0.2077 | train deviance: 0.3074 | val deviance: 0.3129
Epoch 250 | -ELBO/n: 0.2076 | train deviance: 0.3075 | val deviance: 0.3130
Epoch 300 | -ELBO/n: 0.2074 | train deviance: 0.3072 | val deviance: 0.3130


In [ ]:
# Write to disk for later uses
import json

with open("plotting_json/poisson_horseshoe_bcann.json", "w") as f:
    json.dump(horseshoe_bcann_history_full, f, indent=2)

## Some sensitivity testing

In [ ]:
# sensitivity of prediction performance on value of tau

tau_sensi_scores = []

for tau in [1e-3, 1e-2, 1e-1, 1.0, 2.0]:
    pyro.clear_param_store()
    set_seed(seed)

    sigma_w = float(np.sqrt(tau))  # prior_scale is a std dev; tau = sigma_w^2
    model = HorseshoePoissonBCANN(
        in_features=48,
        neural_net_params={"hidden_activation_nodes": network_architecture,
                           "hidden_activations": nn.Tanh},
                           tau_0= tau).to(device)
    guide = build_guide_horseshoe_bcann(model, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)
    bcann_history = poisson_bcann_svi_training_loop(
        model, guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=5, patience=10,
        random_state=seed, device=device, kl_scheme = "uniform")

    tau_sensi_scores.append({"tau": tau, **bcann_history})


Epoch 1 | -ELBO/n: 2.0754 | train deviance: 0.3150 | val deviance: 0.3201
Epoch 25 | -ELBO/n: 0.2163 | train deviance: 0.3146 | val deviance: 0.3198
Epoch 50 | -ELBO/n: 0.2145 | train deviance: 0.3146 | val deviance: 0.3198
Epoch 75 | -ELBO/n: 0.2122 | train deviance: 0.3146 | val deviance: 0.3198


In [ ]:
# sensitivity of prediction performance on value of slab_scale

slab_scale_sensi_scores = []

for ss in [0.5, 1.0, 2.0, 4.0]:
    pyro.clear_param_store()
    set_seed(seed)

    model = HorseshoePoissonBCANN(
        in_features=48,
        neural_net_params={"hidden_activation_nodes": network_architecture,
                           "hidden_activations": nn.Tanh},
                           slab_scale= ss).to(device)
    guide = build_guide_horseshoe_bcann(model, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)
    bcann_history = poisson_bcann_svi_training_loop(
        model, guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=5, patience=10,
        random_state=seed, device=device, kl_scheme = "uniform")

    slab_scale_sensi_scores.append({"slab_scale": ss, **bcann_history})


Epoch 1 | -ELBO/n: 344.1691 | val deviance: 145.4771
Epoch 25 | -ELBO/n: 0.2126 | val deviance: 0.3203
Epoch 50 | -ELBO/n: 0.2113 | val deviance: 0.3207
Epoch 75 | -ELBO/n: 0.2101 | val deviance: 0.3211
Epoch 100 | -ELBO/n: 0.2096 | val deviance: 0.3203
Epoch 125 | -ELBO/n: 0.2096 | val deviance: 0.3202
Epoch 150 | -ELBO/n: 0.2095 | val deviance: 0.3201
Epoch 175 | -ELBO/n: 0.2097 | val deviance: 0.3199
Epoch 200 | -ELBO/n: 0.2098 | val deviance: 0.3201
Early stopping at epoch 205. Best val deviance 0.3198 at epoch 155
Epoch 1 | -ELBO/n: 316.1912 | val deviance: 458.6653
Epoch 25 | -ELBO/n: 0.2140 | val deviance: 0.3227
Epoch 50 | -ELBO/n: 0.2120 | val deviance: 0.3200
Epoch 75 | -ELBO/n: 0.2103 | val deviance: 0.3213
Early stopping at epoch 85. Best val deviance 0.3199 at epoch 35
Epoch 1 | -ELBO/n: 290.9566 | val deviance: 140.5390
Epoch 25 | -ELBO/n: 0.2143 | val deviance: 0.3210
Epoch 50 | -ELBO/n: 0.2129 | val deviance: 0.3267
Epoch 75 | -ELBO/n: 0.2122 | val deviance: 0.3200
Epoc

In [ ]:
# sensitivity of prediction performance on value of slab_df

slab_df_sensi_scores = []

for df in [2.0, 4.0, 8.0, 25.0]:
    pyro.clear_param_store()
    set_seed(seed)

    model = HorseshoePoissonBCANN(
        in_features=48,
        neural_net_params={"hidden_activation_nodes": network_architecture,
                           "hidden_activations": nn.Tanh},
                           slab_df= df).to(device)
    guide = build_guide_horseshoe_bcann(model, X_train[:256], y_train[:256], glm_mle=glm_mle, rank=10)
    bcann_history = poisson_bcann_svi_training_loop(
        model, guide, train_loader, val_loader,
        n_epochs=300, lr=5e-3, val_every=5, patience=10,
        random_state=seed, device=device, kl_scheme = "uniform")

    slab_scale_sensi_scores.append({"slab_df": df, **bcann_history})


Epoch 1 | -ELBO/n: 290.7281 | val deviance: 140.7248
Epoch 25 | -ELBO/n: 0.2145 | val deviance: 0.3202
Epoch 50 | -ELBO/n: 0.2130 | val deviance: 0.3201
Epoch 75 | -ELBO/n: 0.2119 | val deviance: 0.3203
Epoch 100 | -ELBO/n: 0.2105 | val deviance: 0.3199
Epoch 125 | -ELBO/n: 0.2109 | val deviance: 0.3220
Epoch 150 | -ELBO/n: 0.2100 | val deviance: 0.3204
Early stopping at epoch 155. Best val deviance 0.3199 at epoch 105
Epoch 1 | -ELBO/n: 290.9566 | val deviance: 140.5390
Epoch 25 | -ELBO/n: 0.2143 | val deviance: 0.3210
Epoch 50 | -ELBO/n: 0.2129 | val deviance: 0.3267
Epoch 75 | -ELBO/n: 0.2122 | val deviance: 0.3200
Epoch 100 | -ELBO/n: 0.2119 | val deviance: 0.3218
Epoch 125 | -ELBO/n: 0.2101 | val deviance: 0.3202
Epoch 150 | -ELBO/n: 0.2104 | val deviance: 0.3219
Early stopping at epoch 160. Best val deviance 0.3199 at epoch 110
Epoch 1 | -ELBO/n: 290.7210 | val deviance: 140.6338
Epoch 25 | -ELBO/n: 0.2144 | val deviance: 0.3236
Epoch 50 | -ELBO/n: 0.2139 | val deviance: 0.3209
E